# Plots

In [ ]:
library(readr)
library(purrr)
library(dplyr)
library(stringr)
library(fuzzyjoin)
library(ggplot2)
library(dplyr)
library(tidyverse)
library(Matrix)
library(reshape2)
library(RColorBrewer)
library(rstatix)
library(emmeans)


In [ ]:
#show all columns
options(repr.matrix.max.cols = Inf,  # show all columns
        repr.matrix.max.rows = 200)  # adjust rows as you like

#dont show col types        
options(readr.show_col_types = FALSE)

## 0. Plot parameters

In [ ]:
parent_dir = c(
"/ceph.groups/mshahbazi.grp/rsakata/HE_Nondeveloping3/output/cellpose_boundary3.2",
"/ceph.groups/mshahbazi.grp/rsakata/HE_Nondeveloping4/output/cellpose_boundary3",
"/ceph.groups/mshahbazi.grp/rsakata/HE_Nondeveloping5/output/cellpose_boundary3",
"/ceph.groups/mshahbazi.grp/rsakata/HE_Nondeveloping6/output/cellpose_boundary3")


sample_sheet_csv = "/ceph.groups/mshahbazi.grp/rsakata/Figures/HumanEmbyos/sample_sheet_HE.csv"
out_dir = "/ceph.groups/mshahbazi.grp/rsakata/Figures/HumanEmbyos/output"

In [ ]:
if (!dir.exists(out_dir)) { 
  dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)
 }

In [ ]:
# define const for visualization
FONT.SIZE <- 7
LABEL.FONT.SIZE <- 7
w <- 2 
h <- 2.5
LINE.W <- 0.5/2.141959

# Set geom defaults globally
update_geom_defaults("line",      list(linewidth = LINE.W))
update_geom_defaults("errorbar",  list(linewidth = LINE.W))
#update_geom_defaults("point",     list(size = LINE.W, stroke = LINE.W))

settheme <- theme_minimal() + 
  theme(
    text = element_text(family = "sans"), 
    panel.background = element_blank(),
    panel.grid.major = element_blank(), 
    panel.grid.minor = element_blank(),
    plot.background = element_blank(),
    axis.ticks = element_line(colour = "black", linewidth = LINE.W),
    axis.ticks.length = unit(0.1, "cm"), 
    axis.line = element_line(linewidth = LINE.W, colour = "black"),
    axis.title = element_text(size = FONT.SIZE),
    axis.text = element_text(colour = "black", size = FONT.SIZE),
    strip.text = element_text(size = FONT.SIZE), 
    strip.text.y.left = element_text(angle = 0, hjust = 1, size = FONT.SIZE),
    legend.position = "right",
    legend.title = element_text(size = FONT.SIZE), 
    legend.text = element_text(size = FONT.SIZE),
    legend.key.size = unit(0.3, "cm"),
    axis.text.x = element_text(colour = "black", angle = 0, size = LABEL.FONT.SIZE),
    title = element_text(size = FONT.SIZE) 
  )

In [ ]:
col_aneu = c("euploid"= "#D4D1B3","monosomy"="#109E9D","trisomy"="#F26B3B","complex"= "#886DB0")

col_condition_2 = c("control" = "#285F63", 
               "reversine" = "#CA4F33", 
               "mosaic"= "#E2A557")

col_cell_cell = c("WT_WT" = "#5E5E5E", 
               "GFP_GFP" = "#86AB30", 
               "GFP_WT"= "#E2A557")

col_cell_cell2 = c("WT-WT" = "#5E5E5E", 
               "WT-Rev" = "#E2A557", 
               "Rev-Rev"= "#EB5951")

col_condition = c("Poor Quality"= "#5E5E5E","Developed"="#86AB30","Rrev_G"="#EB5951", "Grev_Rrev"="#F0A329")

col_GFP = c("TRUE"= "#86AB30","FALSE"="#8d8d8dff")
col_state = c("Developed"= "#4674b8",'Poor Quality'="#8d8d8dff")
col_timepoint = c("D4"= "#109E9D","D6"="#285F63")


## 1. Extract summary files

In [ ]:
files <- list.files(
  path = parent_dir, 
  pattern = "_summary\\.csv$", 
  recursive = TRUE, 
  full.names = TRUE
)
files

In [ ]:
files <- list.files(
  path = parent_dir, 
  pattern = "_summary\\.csv$", 
  recursive = TRUE, 
  full.names = TRUE
)

data_merged <- files %>%
  set_names() %>%
  map_dfr(read_csv, .id = "filename")


In [ ]:
# add columns to identify the samples and image
data_merged <- data_merged %>%
  mutate(
    EXP = str_extract(filename, "(?<=EXP).{2}"), 
    image = sub("_data\\.csv$", "", basename(filename)),
    sample = sub("_.*", "", image)
  )

head(data_merged)

In [ ]:
unique(data_merged$sample)

In [ ]:
#merge with sample sheet
sample_sheet <- read_csv(sample_sheet_csv, show_col_types = FALSE)
sample_sheet$sample <- as.character(sample_sheet$sample_name)
head(sample_sheet)

In [ ]:

merged_df <- data_merged %>%
  left_join(sample_sheet, by = "sample")   # keeps all rows from df1

merged_df$EXP = merged_df$EXP.y

In [ ]:
merged_df = merged_df %>% filter(Use == "Yes")
unique(merged_df$sample)
tbl <- merged_df %>%
  group_by(condition) %>%
  summarise(n_images = n_distinct(sample_name), .groups = "drop")

tbl

In [ ]:
head(merged_df)

In [ ]:
tbl <- merged_df %>%
  group_by(EXP, condition, sample_name) %>%
  summarise(n_images = n_distinct(sample_name), .groups = "drop")

tbl

In [ ]:
# order_sample <- c(
#   "D4_GR",  "D4_GrevRrev", "D4_GrevR", "D4_RrevG",
#   "D6_GR_D", "D6_GR_F", "D6_GrevRrev_D", "D6_GrevRrev_F", 
#   "D6_GrevR_D", "D6_GrevR_F", "D6_RrevG_D", "D6_RrevG_F")
# merged_df <- merged_df %>%
#   mutate(sample_name = factor(sample_name, levels = order_sample))

## Plots

### A) Normalised intensity (mean of mean per image)

In [ ]:
head(merged_df)

In [ ]:
plot_data = merged_df

In [ ]:
unique(merged_df$condition)

In [ ]:
plot_data = plot_data %>% filter(channel%in% c("ECAD", "DAPI"))

In [ ]:
w <- 8
h <- 2
title = "Normalised_intensity_meanperimage"
options(repr.plot.width=w, repr.plot.height=h)

p <- ggplot(plot_data, aes(x = distance_um, y = mean, color = condition,fill = condition,  group = image )) +
  
  # The main average line
  geom_line(linewidth = 0.5) +
  
  # Labels and Theme
  labs(
    title = title ,
    x = "Distance from Peak",
    y = "Normalized Intensity",
    color = "Cell Pair Class",
    fill  = "Cell Pair Class"
  )+ facet_grid(channel~sample_name, scales = "free_y") +
  settheme +
 # theme(legend.position = "none")+ 
  scale_fill_manual(values = col_condition)

ggsave(plot = p, filename = sprintf("%s/A_%s.pdf",out_dir, title), w = w, h = h)
p

### B) Background substracted normalised background 

In [ ]:
head(merged_df)

In [ ]:
# background substract
plot_data <- merged_df %>%
  filter(channel %in% c( "ECAD")) %>%

  # Calculate Background (Minimum of the mean profile)
  # We group by sample and channel (ignoring distance) to find the lowest point of the line
  group_by(sample_name, condition, channel) %>%
  mutate(
    background = min(mean, na.rm = TRUE),
    corrected_mean = mean - background
  ) %>%
  ungroup() %>%

  # 4. Define Limits for plotting
  # Note: Substracting a constant background does NOT change the SD/SE width, 
  # so we apply the error bars to the NEW corrected mean.
  mutate(
    ymin = corrected_mean - sem_within_image, 
    ymax = corrected_mean + sem_within_image
  )

In [ ]:
w <- 8
h <- 2
title = "Normalised_intensity-background"
options(repr.plot.width=w, repr.plot.height=h)

plot <- ggplot(plot_data, aes(x = distance_um, y = corrected_mean, fill = condition, color = condition)) +
  
  # Add the vertical dashed line at 0 (center)
  geom_vline(xintercept = 0, linetype = "dashed", color = "gray50", linewidth = 0.5) +
  
  # The shaded error band (Ribbon)
  geom_ribbon(aes(ymin = ymin, ymax = ymax), alpha = 0.2, color = NA) +
  
  # The main average line
  geom_line(linewidth = 0.5) +
  
  # Labels and Theme
  labs(
    title = title ,
    x = "Distance from Peak",
    y = "Normalized Intensity",
    color = "Cell Pair Class",
    fill  = "Cell Pair Class"
  )+ facet_grid( channel ~ sample_name, scales = "free_y") +
  settheme +
  theme(legend.position = "none")+ 
  scale_fill_manual(values = col_condition)+ 
  scale_color_manual(values = col_condition)

ggsave(plot = plot , filename = sprintf("%s/B_%s.pdf",out_dir, title), w = w, h = h)
plot 

In [ ]:
w <- 4
h <- 2
title = "Normalised_intensity-background"
options(repr.plot.width=w, repr.plot.height=h)

plot <- ggplot(plot_data, aes(x = distance_um, y = corrected_mean, fill = condition, color = condition, group = sample_name)) +
  
  # Add the vertical dashed line at 0 (center)
  geom_vline(xintercept = 0, linetype = "dashed", color = "gray50", linewidth = 0.5) +
  
  # The shaded error band (Ribbon)
  geom_ribbon(aes(ymin = ymin, ymax = ymax), alpha = 0.2, color = NA) +
  
  # The main average line
  geom_line(linewidth = 0.5) +
  
  # Labels and Theme
  labs(
    title = title ,
    x = "Distance from Peak",
    y = "Normalized Intensity",
    color = "Cell Pair Class",
    fill  = "Cell Pair Class"
  )+ facet_grid( ~ condition, scales = "free_y") +
  settheme +
  theme(legend.position = "none")+ 
  scale_fill_manual(values = col_condition)+ 
  scale_color_manual(values = col_condition)

ggsave(plot = plot , filename = sprintf("%s/B_%s.pdf",out_dir, title), w = w, h = h)
plot 

### E) Failed vs developed

In [ ]:
head(plot_data)

In [ ]:
# failed vs developed
plot_data2 <- plot_data %>%
  filter(channel %in% c( "ECAD"))%>%

  # mean of each state
  group_by(condition ,distance_um, channel) %>%
  summarise(
    mean_int = mean(corrected_mean	, na.rm = TRUE),
    sd_int   = sd(corrected_mean	, na.rm = TRUE),
    n        = n(),
    se_int   = sd_int / sqrt(n), # Standard Error
    .groups  = "drop"
  )%>%

  # 4. Define Limits for plotting
  # Note: Substracting a constant background does NOT change the SD/SE width, 
  # so we apply the error bars to the NEW corrected mean.
  mutate(
    ymin =  mean_int - se_int , 
    ymax =  mean_int + se_int 
  )

In [ ]:
unique(plot_data2$condition)

In [ ]:
w <- 2.5
h <- 1.8
title = "failed_developed"
options(repr.plot.width=w, repr.plot.height=h)

plot <- ggplot(plot_data2, aes(x = distance_um, y =  mean_int , fill = condition, color = condition)) +
  
  # Add the vertical dashed line at 0 (center)
  geom_vline(xintercept = 0, linetype = "dashed", color = "gray50", linewidth = 0.5) +
  
  # The shaded error band (Ribbon)
  geom_ribbon(aes(ymin = ymin, ymax = ymax), alpha = 0.2, color = NA) +
  
  # The main average line
  geom_line(linewidth = 0.5) +
  
  # Labels and Theme
  labs(
    title = title ,
    x = "Distance from Peak",
    y = "Normalized Intensity",
    color = "state",
    fill  = "state"
  )+ facet_grid( ~ channel, scales = "free_y") +
  settheme + 
  scale_fill_manual(values = col_state)+ 
  scale_color_manual(values = col_state)

ggsave(plot = plot , filename = sprintf("%s/E_%s.pdf",out_dir, title), w = w, h = h)
plot 

In [ ]:
w <- 1.2
h <- 2.1
title = "condition_dots"
options(repr.plot.width=w, repr.plot.height=h)

plot_data3 = plot_data %>% filter(distance_um == 0)

p = ggplot(plot_data3, aes(x = condition, y = corrected_mean)) +  # dots for each file
  stat_summary(
    fun = mean, 
    geom = "bar",
    position = position_dodge(width = 0.75),
    fill = "grey", alpha = 0.4, width = 0.6) +   # error bars
  geom_jitter(
    aes(color = condition),
    position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
    size = 0.5, alpha = 0.8
  )  +  # average bar
  stat_summary(
    fun.data = mean_se, 
    geom = "errorbar",
    position = position_dodge(width = 0.75),
    width = 0.1, 
    color = "black")+
    scale_y_continuous(limits = c(0, NA), 
    expand = expansion(mult = c(0, 0.1)))+
  labs(
    title =title,
    y = "Normalized Intensity",
    x = ""
  )+ facet_wrap(~channel, nrow= 1, scales = "free_y") + settheme+
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    legend.position = "none"
  ) +  
  scale_fill_manual(values = col_state)+ 
  scale_color_manual(values =col_state)
              

ggsave(plot = p  , filename = sprintf("%s/E2_%s.pdf",out_dir, title), w = w, h = h)
p 

In [ ]:
library(dplyr)
library(tidyr)
library(purrr)

check_test <- function(data, group_var = "condition", value_var = "corrected_mean",
                        conditions = c("Developed", "Faiiled"), alpha = 0.05) {

  d <- data %>%
    filter(.data[[group_var]] %in% conditions) %>%
    droplevels()

  d %>%
    #group_by(state) %>%
    group_modify(~ {
      g <- split(.x[[value_var]], .x[[group_var]])
      g <- g[conditions]                       # keep the two groups in order

      # need at least 3 non-NA points per group for Shapiro
      n_ok <- all(sapply(g, function(x) sum(!is.na(x)) >= 3))

      if (!n_ok) {
        return(tibble(
          n1 = sum(!is.na(g[[1]])), n2 = sum(!is.na(g[[2]])),
          shapiro_p1 = NA_real_, shapiro_p2 = NA_real_,
          levene_p = NA_real_, normal = NA,
          recommended = "too few points (use Wilcoxon / be cautious)"
        ))
      }

      # normality per group
      sp1 <- shapiro.test(g[[1]])$p.value
      sp2 <- shapiro.test(g[[2]])$p.value
      normal <- (sp1 > alpha) & (sp2 > alpha)

      # equal-variance check (F-test; swap for car::leveneTest if preferred)
      var_p <- tryCatch(var.test(g[[1]], g[[2]])$p.value, error = function(e) NA_real_)

      rec <- if (normal) {
        if (!is.na(var_p) && var_p > alpha) "Student t-test (var.equal = TRUE)"
        else "Welch t-test"
      } else {
        "Wilcoxon rank-sum test"
      }

      tibble(
        n1 = sum(!is.na(g[[1]])), n2 = sum(!is.na(g[[2]])),
        shapiro_p1 = sp1, shapiro_p2 = sp2,
        levene_p = var_p, normal = normal,
        recommended = rec
      )
    }) %>%
    ungroup()
}

# usage
check_test(plot_data3)

In [ ]:
t_test <- plot_data3  %>%
  t_test(
    corrected_mean ~ condition,
   #ref.group = "G_R",
    p.adjust.method = "holm"
  ) %>%
  mutate(
    p_use = if ("p.adj" %in% names(.)) p.adj else p,   # fall back to raw p
    stars = case_when(
      p_use < 0.0001 ~ "****",
      p_use < 0.001  ~ "***",
      p_use < 0.01   ~ "**",
      p_use < 0.05   ~ "*",
      TRUE           ~ "ns"
    )
  )

wilcox_res